Import necessary packages.

In [1]:
import os
import numpy as np
import torch as T
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.distributions import Normal

import warnings
warnings.filterwarnings('ignore')

import gymnasium as gym


from gymnasium.wrappers import RecordVideo
from tqdm.notebook import trange
from IPython.display import Video

# compatibility fix (NumPy 2.x removed np.bool8)
np.bool8 = np.bool_


In [3]:
class ReplayBuffer:
    def __init__(self, buffer_size, state_dims, action_dims):
        self.buffer_size = buffer_size
        self.ptr = 0
        self.is_full = False

        # Initialize buffers
        self.states = np.zeros((buffer_size, *state_dims), dtype=np.float32)
        self.next_states = np.zeros((buffer_size, *state_dims), dtype=np.float32)
        self.actions = np.zeros((buffer_size, *action_dims), dtype=np.float32)
        self.rewards = np.zeros(buffer_size, dtype=np.float32)
        self.dones = np.zeros(buffer_size, dtype=np.float32)

    def store_transition(self, state, action, reward, state_, done):
        self.states[self.ptr] = state
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.next_states[self.ptr] = state_
        self.dones[self.ptr] = done

        self.ptr += 1
        if self.ptr >= self.buffer_size:
            self.ptr = 0
            self.is_full = True

    def load_batch(self, batch_size):
        max_mem = self.buffer_size if self.is_full else self.ptr
        batch_indices = np.random.choice(max_mem, batch_size, replace=False)

        states = T.tensor(self.states[batch_indices]).float()
        actions = T.tensor(self.actions[batch_indices]).float()
        rewards = T.tensor(self.rewards[batch_indices]).float().unsqueeze(1)
        states_ = T.tensor(self.next_states[batch_indices]).float()
        done = T.tensor(self.dones[batch_indices]).float().unsqueeze(1)

        return states, actions, rewards, states_, done


In [2]:
device = T.device('cuda' if T.cuda.is_available() else 'cpu')


class Critic(nn.Module):
    def __init__(self, beta, state_dims, action_dims, fc1_dims, fc2_dims,
                 name='Critic', ckpt_dir='tmp'):
        super(Critic, self).__init__()
        # Save args
        self.state_dims = state_dims
        self.action_dims = action_dims
        self.fc1_dims = fc1_dims
        self.fc2_dims = fc2_dims
        self.name = name
        self.ckpt_dir = ckpt_dir
        self.ckpt_path = os.path.join(ckpt_dir, name + '.pth')

        # Define layers
        self.fc1 = nn.Linear(state_dims[0] + action_dims[0], fc1_dims)
        self.fc2 = nn.Linear(fc1_dims, fc2_dims)
        self.q = nn.Linear(fc2_dims, 1)

        # Optimizer
        self.optimizer = optim.Adam(self.parameters(), lr=beta)
        self.to(device)

    def forward(self, state, action):
        x = T.cat([state, action], dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        q = self.q(x)
        return q

    def save_checkpoint(self):
        T.save(self.state_dict(), self.ckpt_path)

    def load_checkpoint(self, gpu_to_cpu=False):
        if gpu_to_cpu:
            self.load_state_dict(T.load(self.ckpt_path, map_location=lambda storage, loc: storage))
        else:
            self.load_state_dict(T.load(self.ckpt_path))


class Actor(nn.Module):
    def __init__(self, alpha, state_dims, action_dims, fc1_dims, fc2_dims,
                 max_action, reparam_noise, name='Actor', ckpt_dir='tmp'):
        super(Actor, self).__init__()
        # Save args
        self.state_dims = state_dims
        self.action_dims = action_dims
        self.fc1_dims = fc1_dims
        self.fc2_dims = fc2_dims
        self.max_action = max_action
        self.reparam_noise = reparam_noise
        self.name = name
        self.ckpt_dir = ckpt_dir
        self.ckpt_path = os.path.join(ckpt_dir, name + '.pth')

        # Layers
        self.fc1 = nn.Linear(state_dims[0], fc1_dims)
        self.fc2 = nn.Linear(fc1_dims, fc2_dims)
        self.mu = nn.Linear(fc2_dims, action_dims[0])
        self.sigma = nn.Linear(fc2_dims, action_dims[0])

        # Optimizer
        self.optimizer = optim.Adam(self.parameters(), lr=alpha)
        self.to(device)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        mu = self.mu(x)
        sigma = F.softplus(self.sigma(x)) + 1e-5  # Ensure positivity
        return mu, sigma

    def sample_normal(self, state, reparameterize=True):
        mu, sigma = self.forward(state)
        dist = Normal(mu, sigma)
        if reparameterize:
            action = dist.rsample()  # reparameterization trick
        else:
            action = dist.sample()
        log_probs = dist.log_prob(action).sum(dim=1, keepdim=True)
        action = T.tanh(action) * self.max_action
        return action, log_probs

    def save_checkpoint(self):
        T.save(self.state_dict(), self.ckpt_path)

    def load_checkpoint(self, gpu_to_cpu=False):
        if gpu_to_cpu:
            self.load_state_dict(T.load(self.ckpt_path, map_location=lambda storage, loc: storage))
        else:
            self.load_state_dict(T.load(self.ckpt_path))


class Value(nn.Module):
    def __init__(self, beta, state_dims, fc1_dims, fc2_dims,
                 name='Value', ckpt_dir='tmp'):
        super(Value, self).__init__()
        # Save args
        self.state_dims = state_dims
        self.fc1_dims = fc1_dims
        self.fc2_dims = fc2_dims
        self.name = name
        self.ckpt_dir = ckpt_dir
        self.ckpt_path = os.path.join(ckpt_dir, name + '.pth')

        # Layers
        self.fc1 = nn.Linear(state_dims[0], fc1_dims)
        self.fc2 = nn.Linear(fc1_dims, fc2_dims)
        self.v = nn.Linear(fc2_dims, 1)

        # Optimizer
        self.optimizer = optim.Adam(self.parameters(), lr=beta)
        self.to(device)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        v = self.v(x)
        return v

    def save_checkpoint(self):
        T.save(self.state_dict(), self.ckpt_path)

    def load_checkpoint(self, gpu_to_cpu=False):
        if gpu_to_cpu:
            self.load_state_dict(T.load(self.ckpt_path, map_location=lambda storage, loc: storage))
        else:
            self.load_state_dict(T.load(self.ckpt_path))


In [3]:
class Agent:
    def __init__(self, gamma, alpha, beta, state_dims, action_dims, max_action,
                 fc1_dim, fc2_dim, memory_size, batch_size, tau, update_period,
                 reward_scale, warmup, reparam_noise_lim, name, ckpt_dir='tmp'):

        # --- Hyperparameters ---
        self.gamma = gamma
        self.alpha = alpha
        self.beta = beta
        self.state_dims = state_dims
        self.action_dims = action_dims
        self.max_action = max_action
        self.fc1_dim = fc1_dim
        self.fc2_dim = fc2_dim
        self.memory_size = memory_size
        self.batch_size = batch_size
        self.tau = tau
        self.update_period = update_period
        self.reward_scale = reward_scale
        self.warmup = warmup
        self.reparam_noise_lim = reparam_noise_lim
        self.ckpt_dir = ckpt_dir

        model_name = f'{name}__gamma_{gamma}__alpha_{alpha}__beta_{beta}__fc1_{fc1_dim}__fc2_{fc2_dim}__bs_{batch_size}__buffer_{memory_size}__update_period_{update_period}__tau_{tau}__'
        self.model_name = model_name
        self.full_path = os.path.join(self.ckpt_dir, self.model_name)
        self.learn_iter = 0

        # --- Replay buffer ---
        self.memory = ReplayBuffer(memory_size, state_dims, action_dims)

        # --- Actor and Critic networks ---
        self.actor = Actor(alpha, state_dims, action_dims, fc1_dim, fc2_dim,
                           max_action, reparam_noise_lim, name='actor', ckpt_dir=ckpt_dir)
        self.critic_1 = Critic(beta, state_dims, action_dims, fc1_dim, fc2_dim,
                               name='critic_1', ckpt_dir=ckpt_dir)
        self.critic_2 = Critic(beta, state_dims, action_dims, fc1_dim, fc2_dim,
                               name='critic_2', ckpt_dir=ckpt_dir)

        # --- Value and target value networks ---
        self.value = Value(beta, state_dims, fc1_dim, fc2_dim, name='value', ckpt_dir=ckpt_dir)
        self.target_value = Value(beta, state_dims, fc1_dim, fc2_dim, name='target_value', ckpt_dir=ckpt_dir)

        # --- Sync target network initially ---
        self.update_target_network(tau=1.0)

    # --- Action selection ---
    def choose_action(self, state, deterministic=False, reparameterize=False):
        state = T.tensor(state, dtype=T.float32).unsqueeze(0).to(device)
        action, _ = self.actor.sample_normal(state, reparameterize=reparameterize)
        return action.cpu().detach().numpy()[0]

    # --- Store transition in replay buffer ---
    def store_transition(self, state, action, reward, state_, done):
        self.memory.store_transition(state, action, reward, state_, done)

    # --- Sample batch from replay buffer ---
    def load_batch(self):
        return self.memory.load_batch(self.batch_size)

    # --- Soft update target network ---
    def update_target_network(self, tau=None):
        tau = self.tau if tau is None else tau
        for target_param, param in zip(self.target_value.parameters(), self.value.parameters()):
            target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)

    # --- Learning step ---
    def learn(self):
        if self.memory.ptr < self.warmup:
            return  # Skip learning until enough samples collected

        states, actions, rewards, states_, dones = self.load_batch()
        states = states.to(device)
        actions = actions.to(device)
        rewards = rewards.to(device)
        states_ = states_.to(device)
        dones = dones.to(device)

        # ---------------- VALUE LOSS ----------------
        # Sample action from actor
        sampled_actions, log_probs = self.actor.sample_normal(states, reparameterize=True)

        # Estimate Q-values
        q1_val = self.critic_1(states, sampled_actions)
        q2_val = self.critic_2(states, sampled_actions)
        min_q = T.min(q1_val, q2_val)

        # Target value
        target_v = min_q - self.alpha * log_probs
        v = self.value(states)
        value_loss = F.mse_loss(v, target_v.detach())
        self.value.optimizer.zero_grad()
        value_loss.backward()
        self.value.optimizer.step()

        # ---------------- ACTOR LOSS ----------------
        new_actions, log_probs = self.actor.sample_normal(states, reparameterize=True)
        q1_new = self.critic_1(states, new_actions)
        q2_new = self.critic_2(states, new_actions)
        min_q_new = T.min(q1_new, q2_new)
        actor_loss = (self.alpha * log_probs - min_q_new).mean()
        self.actor.optimizer.zero_grad()
        actor_loss.backward()
        self.actor.optimizer.step()

        # ---------------- CRITIC LOSS ----------------
        # Compute Q target
        with T.no_grad():
            v_next = self.target_value(states_)
            q_target = self.reward_scale * rewards + self.gamma * (1 - dones) * v_next

        q1_current = self.critic_1(states, actions)
        q2_current = self.critic_2(states, actions)
        critic_1_loss = F.mse_loss(q1_current, q_target)
        critic_2_loss = F.mse_loss(q2_current, q_target)

        self.critic_1.optimizer.zero_grad()
        critic_1_loss.backward()
        self.critic_1.optimizer.step()

        self.critic_2.optimizer.zero_grad()
        critic_2_loss.backward()
        self.critic_2.optimizer.step()

        # ---------------- TARGET NETWORK UPDATE ----------------
        if self.learn_iter % self.update_period == 0:
            self.update_target_network()

        # Increment learning iteration counter
        self.learn_iter += 1
    def save_model(self):
        print('... saving checkpoint ...')
        self.actor.save_checkpoint()
        self.critic_1.save_checkpoint()
        self.critic_2.save_checkpoint()
        self.value.save_checkpoint()
        self.target_value.save_checkpoint()

    def load_model(self, gpu_to_cpu=False):
        print('... loading checkpoint ...')
        self.actor.load_checkpoint(gpu_to_cpu=gpu_to_cpu)
        self.critic_1.load_checkpoint(gpu_to_cpu=gpu_to_cpu)
        self.critic_2.load_checkpoint(gpu_to_cpu=gpu_to_cpu)
        self.value.load_checkpoint(gpu_to_cpu=gpu_to_cpu)
        self.target_value.load_checkpoint(gpu_to_cpu=gpu_to_cpu)



In [ ]:
# --- Configuration parameters ---
env_name = "HalfCheetah-v4"
env = gym.make(env_name, render_mode="rgb_array")
dir = 'tmp'
n_games = 500  # Number of episodes to train

gamma = 0.99
alpha = 3e-4
beta = 3e-4
fc1_dim = 256
fc2_dim = 256
memory_size = 1000000
batch_size = 256
tau = 0.005
update_period = 2
reward_scale = 2.0
warmup = 1000
reparam_noise_lim = 1e-6
record_video = True


In [ ]:
env_name = "HalfCheetah-v4"
eval_env = gym.make(env_name, render_mode="rgb_array")

state_dims = eval_env.observation_space.shape
action_dims = eval_env.action_space.shape
max_action = eval_env.action_space.high[0]

agent_checkpoint_dir = "agent"

eval_agent = Agent(
    gamma=0.99,
    alpha=3e-4,
    beta=3e-4,
    state_dims=state_dims,
    action_dims=action_dims,
    max_action=max_action,
    fc1_dim=256,
    fc2_dim=256,
    memory_size=1000000,
    batch_size=256,
    tau=0.005,
    update_period=2,
    reward_scale=2.0,
    warmup=1000,
    reparam_noise_lim=1e-6,
    name="SAC",
    ckpt_dir=agent_checkpoint_dir,
)

eval_agent.ckpt_dir = agent_checkpoint_dir
eval_agent.actor.ckpt_dir = agent_checkpoint_dir
eval_agent.actor.ckpt_path = os.path.join(agent_checkpoint_dir, 'actor.pth')
eval_agent.critic_1.ckpt_dir = agent_checkpoint_dir
eval_agent.critic_1.ckpt_path = os.path.join(agent_checkpoint_dir, 'critic_1.pth')
eval_agent.critic_2.ckpt_dir = agent_checkpoint_dir
eval_agent.critic_2.ckpt_path = os.path.join(agent_checkpoint_dir, 'critic_2.pth')
eval_agent.value.ckpt_dir = agent_checkpoint_dir
eval_agent.value.ckpt_path = os.path.join(agent_checkpoint_dir, 'value.pth')
eval_agent.target_value.ckpt_dir = agent_checkpoint_dir
eval_agent.target_value.ckpt_path = os.path.join(agent_checkpoint_dir, 'target_value.pth')

model_files = ['actor.pth', 'critic_1.pth', 'critic_2.pth', 'value.pth', 'target_value.pth']
for file in model_files:
    file_path = os.path.join(agent_checkpoint_dir, file)
    exists = os.path.exists(file_path)

eval_agent.load_model(gpu_to_cpu=False)

video_output_dir = "agent_videos"
os.makedirs(video_output_dir, exist_ok=True)

eval_env = RecordVideo(
    eval_env,
    video_folder=video_output_dir,
    episode_trigger=lambda x: True,
    name_prefix="sac_trained_agent"
)


In [ ]:
def evaluate_agent_with_video(env, agent, num_episodes=3, max_steps=1000):
    """
    Evaluate the agent and return performance metrics while recording video
    """
    episode_rewards = []
    episode_lengths = []
    all_rewards = []

    for episode in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        episode_length = 0
        episode_rewards_list = []

        for step in range(max_steps):

            state_tensor = T.tensor([state]).float().to(device)

            with T.no_grad():
                mu, _ = agent.actor.forward(state_tensor)
                action = T.tanh(mu) * agent.max_action
                action = action.cpu().numpy()[0]
              

            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            episode_reward += reward
            episode_length += 1
            episode_rewards_list.append(reward)

            state = next_state

            if done:
                break

        episode_rewards.append(episode_reward)
        episode_lengths.append(episode_length)
        all_rewards.extend(episode_rewards_list)

        print(f"Episode {episode + 1}: Reward = {episode_reward:.2f}, Length = {episode_length}")

    return {
        'episode_rewards': episode_rewards,
        'episode_lengths': episode_lengths,
        'all_rewards': all_rewards,
        'mean_reward': np.mean(episode_rewards),
        'std_reward': np.std(episode_rewards),
        'mean_length': np.mean(episode_lengths),
        'std_length': np.std(episode_lengths),
        'total_steps': sum(episode_lengths)
    }

In [ ]:

eval_results = evaluate_agent_with_video(
    eval_env, 
    eval_agent, 
    num_episodes=3, 
    max_steps=1000, 
    deterministic=True
)

eval_env.close()

